# Lab B — Agentic Contract Intelligence Cycle

## End-to-end architecture

```text
                         USER / SCHEDULED TRIGGER
                                   │
                                   ▼
                         ┌───────────────────┐
                         │ SUPERVISOR AGENT  │
                         │ plan + routing    │
                         └─────────┬─────────┘
                                   │
                                   ▼
                         ┌───────────────────┐
                         │ CONTRACT AGENT    │
                         │                   │
                         │ structured facts  │
                         │ + clause evidence │
                         └─────────┬─────────┘
                                   │
                ┌──────────────────┴──────────────────┐
                ▼                                     ▼
      query_contract_data(...)            retrieve_contract_evidence(...)
      TOOL span                           RETRIEVER span
                │                                     │
                └──────────────────┬──────────────────┘
                                   ▼
                         ┌───────────────────┐
                         │ RISK AGENT        │
                         │ assess exposure   │
                         │ using evidence    │
                         └─────────┬─────────┘
                                   │
                                   ▼
                         ┌───────────────────┐
                         │ ACTION AGENT      │
                         │ proposed followup │
                         └─────────┬─────────┘
                                   │
                                   ▼
                        PRE-DEPLOYMENT EVALS
                                   │
               ┌───────────────────┼────────────────────┐
               │                   │                    │
               ▼                   ▼                    ▼
     RetrievalGroundedness  RetrievalRelevance    Correctness
               │                   │                    │
               └───────────────────┼────────────────────┘
                                   ▼
                        RetrievalSufficiency
                                   │
                                   ▼
                         DEPLOYMENT GATE
                                   │
                                   ▼
                         HUMAN APPROVAL
                                   │
                         ┌─────────┴─────────┐
                         ▼                   ▼
                      APPROVE              REJECT
                         │
                         ▼
                    ACTION QUEUE
                         │
                         ▼
                 FOLLOW-UP EXECUTION
```

## Objective

Build the entire contract agentic cycle on top of **Lab A — Contract Intelligence Foundations**:

**Supervisor → Contract Agent → Risk Agent → Action Agent → Human Approval → Action Queue**

The POC deliberately separates:
- **structured contract facts** from the enterprise contract view,
- **contract evidence** from clause retrieval,
- **reasoning** from action execution,
- **agent quality evaluation** from deployment,
- **AI recommendation** from **human authorization**.

## Evaluation strategy before deployment

This notebook uses MLflow GenAI evaluation with:

- `RetrievalGroundedness()` — checks whether the final response is supported by retrieved evidence.
- `RetrievalRelevance()` — checks whether retrieved contract clauses are relevant to the question.
- `Correctness()` — compares the response with expected facts.
- `RetrievalSufficiency()` — checks whether retrieval contains enough evidence to support the expected facts.

All retrieval calls are traced with an MLflow `RETRIEVER` span so the RAG judges can inspect the evidence.

## Free Edition approach

- Reuses Delta tables created by Lab A.
- Uses `ai_query` when available for supervisor/risk/action reasoning.
- Uses deterministic fallbacks if model access is unavailable.
- Uses MLflow tracing for the complete agent trajectory.
- Uses a development override only if evaluation infrastructure is unavailable; production readiness remains gated.


## Section 1 — Connect to Lab A Assets

In [0]:
# Discover the Lab A tables and define the new approval and action-queue tables.
row = spark.sql(
    "SELECT current_catalog() AS catalog_name, current_schema() AS schema_name"
).first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]
PREFIX = "contract_intel_poc"

SUPPLIER_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_suppliers"
CONTRACT_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_contracts"
CLAUSE_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_clauses"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_enterprise_gold"
VIEW_NAME = f"{CATALOG}.{SCHEMA}.{PREFIX}_enterprise_contract_view"

APPROVAL_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_agent_approvals"
ACTION_QUEUE_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_agent_action_queue"

LLM_ENDPOINT = "system.ai.meta-llama-3-3-70b-instruct"
USE_LLM_REASONING = True

required_tables = [
    SUPPLIER_TABLE,
    CONTRACT_TABLE,
    CLAUSE_TABLE,
    GOLD_TABLE,
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_tables:
    raise RuntimeError(
        "Lab A must be run first. Missing tables: "
        + ", ".join(missing_tables)
    )

print("Lab A foundation detected.")
print("Enterprise view:", VIEW_NAME)


## Section 2 — Configure MLflow Tracing and GenAI Evaluation

In [0]:
# Configure Databricks MLflow tracking and import the tracing and evaluation APIs used by the agent.
import mlflow
import pandas as pd
import numpy as np
import json
import re
import uuid
from datetime import datetime

from mlflow.entities import SpanType, Document
from mlflow.genai.scorers import (
    RetrievalGroundedness,
    RetrievalRelevance,
    Correctness,
    RetrievalSufficiency,
)

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/contract-intelligence-agentic-poc")

print("MLflow version:", mlflow.__version__)
print("GenAI scorers imported successfully.")


## Section 3 — Create Human-Approval and Action-Queue Tables

The AI proposes actions.

It does **not** directly execute enterprise follow-up.

The proposal first enters an approval table. Only an approved action can move into the action queue.


In [0]:
# Create persistent Delta tables for human approval decisions and approved follow-up actions.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {APPROVAL_TABLE} (
    approval_id STRING,
    action_id STRING,
    contract_id STRING,
    supplier_name STRING,
    proposed_action STRING,
    priority STRING,
    rationale STRING,
    evidence STRING,
    requested_at TIMESTAMP,
    approval_status STRING,
    approver STRING,
    reviewed_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ACTION_QUEUE_TABLE} (
    action_id STRING,
    approval_id STRING,
    contract_id STRING,
    supplier_name STRING,
    action_type STRING,
    proposed_action STRING,
    priority STRING,
    rationale STRING,
    evidence STRING,
    queue_status STRING,
    queued_at TIMESTAMP,
    completed_at TIMESTAMP
)
USING DELTA
""")

print("Approval table:", APPROVAL_TABLE)
print("Action queue table:", ACTION_QUEUE_TABLE)


## Section 4 — Build the Structured Contract Tool

In [0]:
# Define the traced structured-data tool used by the Contract Agent.
from pyspark.sql import functions as F

@mlflow.trace(name="query_contract_data", span_type=SpanType.TOOL)
def query_contract_data(
    intent: str,
    contract_id: str = None,
    supplier_id: str = None,
    days: int = 90,
    limit: int = 20
):
    df = spark.table(GOLD_TABLE)

    if contract_id:
        df = df.filter(F.col("contract_id") == contract_id)

    if supplier_id:
        df = df.filter(F.col("supplier_id") == supplier_id)

    if intent == "contract_detail":
        result_df = df

    elif intent == "high_risk":
        result_df = (
            df.filter(F.col("risk_severity") == "HIGH")
            .orderBy("next_milestone_date")
        )

    elif intent == "payments_due":
        result_df = (
            df.filter(F.col("next_payment_due_date").isNotNull())
            .filter(
                F.col("next_payment_due_date")
                <= F.date_add(F.current_date(), int(days))
            )
            .orderBy("next_payment_due_date")
        )

    elif intent == "expiring":
        result_df = (
            df.filter(
                (F.col("days_to_expiry") >= 0)
                & (F.col("days_to_expiry") <= int(days))
            )
            .orderBy("days_to_expiry")
        )

    else:
        result_df = df.orderBy("contract_id")

    rows = [
        row.asDict(recursive=True)
        for row in result_df.limit(int(limit)).collect()
    ]

    return rows


## Section 5 — Build the Contract Evidence Retriever

In [0]:
# Build the local clause index used by the traced contract evidence retriever.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

clause_pdf = spark.table(CLAUSE_TABLE).toPandas().reset_index(drop=True)

retrieval_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

clause_matrix = retrieval_vectorizer.fit_transform(
    clause_pdf["clause_text"].fillna("")
)

print("Contract clause chunks indexed:", len(clause_pdf))


In [0]:
# Define a RETRIEVER span that returns relevant contract clauses and exposes MLflow Document objects to the RAG judges.
@mlflow.trace(
    name="retrieve_contract_evidence",
    span_type=SpanType.RETRIEVER
)
def retrieve_contract_evidence(
    query: str,
    contract_id: str = None,
    top_k: int = 4
):
    candidate_pdf = clause_pdf.copy()

    if contract_id:
        candidate_pdf = candidate_pdf[
            candidate_pdf["contract_id"] == contract_id
        ].copy()

    if candidate_pdf.empty:
        return []

    candidate_indices = candidate_pdf.index.to_list()
    query_vector = retrieval_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        clause_matrix[candidate_indices]
    ).flatten()

    candidate_pdf["relevance_score"] = scores

    result_pdf = (
        candidate_pdf
        .sort_values("relevance_score", ascending=False)
        .head(int(top_k))
    )

    documents = [
        Document(
            page_content=row["clause_text"],
            metadata={
                "doc_uri": row["document_name"],
                "contract_id": row["contract_id"],
                "clause_type": row["clause_type"],
                "chunk_id": row["chunk_id"],
                "relevance_score": float(row["relevance_score"]),
            },
            id=row["chunk_id"],
        )
        for _, row in result_pdf.iterrows()
    ]

    active_span = mlflow.get_current_active_span()
    if active_span is not None:
        active_span.set_outputs(documents)

    return [
        {
            "chunk_id": doc.id,
            "contract_id": doc.metadata["contract_id"],
            "clause_type": doc.metadata["clause_type"],
            "score": doc.metadata["relevance_score"],
            "text": doc.page_content,
        }
        for doc in documents
    ]


## Section 6 — LLM Helper with Free Edition Fallback

In [0]:
# Define a traced LLM helper that calls ai_query when available and returns a fallback response when it is not.
def _clean_json_text(value):
    text = (value or "").strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

@mlflow.trace(name="call_databricks_llm", span_type=SpanType.LLM)
def call_databricks_llm(prompt: str):
    if not USE_LLM_REASONING:
        return None

    try:
        endpoint = LLM_ENDPOINT.replace("'", "''")
        prompt_sql = prompt.replace("'", "''")

        return spark.sql(
            f"SELECT ai_query('{endpoint}', '{prompt_sql}') AS response"
        ).first()["response"]

    except Exception as e:
        print("LLM call unavailable; deterministic reasoning will be used.")
        print(type(e).__name__, str(e)[:300])
        return None


## Section 7 — Supervisor Agent

The Supervisor converts the user request into a plan:
- identify the contract if explicitly mentioned,
- select the structured-data intent,
- decide whether evidence retrieval is required,
- route to the downstream agents.


In [0]:
# Define the Supervisor Agent that plans the downstream contract, risk, and action-agent work.
@mlflow.trace(name="supervisor_agent", span_type=SpanType.AGENT)
def supervisor_agent(question: str):
    contract_match = re.search(
        r"CTR-\d{4}-\d{3}",
        question,
        flags=re.I
    )

    contract_id = (
        contract_match.group(0).upper()
        if contract_match
        else None
    )

    q = question.lower()

    if contract_id:
        intent = "contract_detail"
    elif "payment" in q or "invoice" in q:
        intent = "payments_due"
    elif "expir" in q or "renew" in q:
        intent = "expiring"
    elif "risk" in q or "attention" in q or "intervention" in q:
        intent = "high_risk"
    else:
        intent = "portfolio"

    plan = {
        "contract_id": contract_id,
        "structured_intent": intent,
        "retrieve_evidence": True,
        "run_risk_agent": True,
        "run_action_agent": True,
    }

    prompt = f"""
You are a supervisor for an enterprise contract intelligence system.
Return ONLY JSON.

User question:
{question}

Current deterministic plan:
{json.dumps(plan)}

Return:
{{
  "contract_id": "contract id or null",
  "structured_intent": "contract_detail|high_risk|payments_due|expiring|portfolio",
  "retrieve_evidence": true,
  "run_risk_agent": true,
  "run_action_agent": true
}}
"""

    response = call_databricks_llm(prompt)

    if response:
        try:
            llm_plan = json.loads(_clean_json_text(response))
            for key in plan:
                if key in llm_plan:
                    plan[key] = llm_plan[key]
        except Exception:
            pass

    return plan


## Section 8 — Contract Agent

The Contract Agent combines:
- governed structured contract facts,
- retrieved clause evidence.

The result becomes the evidence package passed to the Risk Agent.


In [0]:
# Define the Contract Agent that combines enterprise-view facts with relevant contractual evidence.
@mlflow.trace(name="contract_agent", span_type=SpanType.AGENT)
def contract_agent(question: str, plan: dict):
    structured_rows = query_contract_data(
        intent=plan["structured_intent"],
        contract_id=plan.get("contract_id"),
        days=120,
        limit=10
    )

    evidence_contract_id = plan.get("contract_id")

    if not evidence_contract_id and structured_rows:
        evidence_contract_id = structured_rows[0].get("contract_id")

    evidence = retrieve_contract_evidence(
        query=question,
        contract_id=evidence_contract_id,
        top_k=4
    )

    return {
        "structured_facts": structured_rows,
        "retrieved_evidence": evidence,
        "focus_contract_id": evidence_contract_id,
    }


## Section 9 — Risk Agent

The Risk Agent must reason **only from structured facts and retrieved contract evidence**.

It produces:
- severity,
- risk type,
- rationale,
- evidence summary.


In [0]:
# Define the Risk Agent that assesses contract exposure using only facts and retrieved evidence.
@mlflow.trace(name="risk_agent", span_type=SpanType.AGENT)
def risk_agent(question: str, contract_context: dict):
    facts = contract_context.get("structured_facts", [])
    evidence = contract_context.get("retrieved_evidence", [])

    fallback_fact = facts[0] if facts else {}

    fallback_result = {
        "contract_id": fallback_fact.get("contract_id"),
        "supplier_name": fallback_fact.get("supplier_name"),
        "risk_type": fallback_fact.get("risk_type", "Unknown"),
        "risk_severity": fallback_fact.get("risk_severity", "UNKNOWN"),
        "risk_rationale": (
            f"Structured contract data identifies "
            f"{fallback_fact.get('risk_type', 'an unspecified risk')} "
            f"with severity {fallback_fact.get('risk_severity', 'UNKNOWN')}."
        ),
        "evidence_summary": (
            evidence[0]["text"]
            if evidence
            else "No contract evidence retrieved."
        ),
    }

    prompt = f"""
You are the Risk Agent in an enterprise contract intelligence system.

Answer ONLY from the supplied structured facts and retrieved evidence.
Do not invent contract terms.

Question:
{question}

STRUCTURED FACTS:
{json.dumps(facts, default=str)}

RETRIEVED EVIDENCE:
{json.dumps(evidence, default=str)}

Return ONLY JSON:
{{
  "contract_id": "string or null",
  "supplier_name": "string or null",
  "risk_type": "string",
  "risk_severity": "LOW|MEDIUM|HIGH|UNKNOWN",
  "risk_rationale": "concise evidence-based explanation",
  "evidence_summary": "most relevant contractual evidence"
}}
"""

    response = call_databricks_llm(prompt)

    if response:
        try:
            parsed = json.loads(_clean_json_text(response))
            return parsed
        except Exception:
            pass

    return fallback_result


## Section 10 — Action Agent

The Action Agent converts the risk assessment into a controlled proposal.

It does not execute the action.

Output:
- action type,
- proposed action,
- owner,
- priority,
- rationale.


In [0]:
# Define the Action Agent that proposes an enterprise follow-up without executing it.
@mlflow.trace(name="action_agent", span_type=SpanType.AGENT)
def action_agent(question: str, contract_context: dict, risk_assessment: dict):
    facts = contract_context.get("structured_facts", [])
    first_fact = facts[0] if facts else {}

    severity = risk_assessment.get("risk_severity", "UNKNOWN")

    priority_map = {
        "HIGH": "P1",
        "MEDIUM": "P2",
        "LOW": "P3",
        "UNKNOWN": "P3",
    }

    fallback_action = (
        first_fact.get("recommended_action")
        or "Request contract owner review and confirm the next required action."
    )

    fallback_result = {
        "contract_id": risk_assessment.get("contract_id"),
        "supplier_name": risk_assessment.get("supplier_name"),
        "action_type": "CONTRACT_FOLLOW_UP",
        "proposed_action": fallback_action,
        "owner": first_fact.get("contract_owner", "Procurement"),
        "priority": priority_map.get(severity, "P3"),
        "rationale": risk_assessment.get("risk_rationale"),
        "evidence": risk_assessment.get("evidence_summary"),
    }

    prompt = f"""
You are the Action Agent in an enterprise contract intelligence system.

Use only the supplied contract facts and risk assessment.
Do not claim that an action has already been executed.

QUESTION:
{question}

CONTRACT FACTS:
{json.dumps(facts, default=str)}

RISK ASSESSMENT:
{json.dumps(risk_assessment, default=str)}

Return ONLY JSON:
{{
  "contract_id": "string or null",
  "supplier_name": "string or null",
  "action_type": "CONTRACT_FOLLOW_UP",
  "proposed_action": "specific follow-up recommendation",
  "owner": "Procurement|Finance|IT|Operations",
  "priority": "P1|P2|P3",
  "rationale": "why this action is needed",
  "evidence": "supporting contract evidence"
}}
"""

    response = call_databricks_llm(prompt)

    if response:
        try:
            parsed = json.loads(_clean_json_text(response))
            return parsed
        except Exception:
            pass

    return fallback_result


## Section 11 — Assemble the Complete Agentic Cycle

The root agent:
1. asks the Supervisor to plan,
2. invokes the Contract Agent,
3. invokes the Risk Agent,
4. invokes the Action Agent,
5. returns a recommendation ready for human approval.

No queue write occurs inside this function.


In [0]:
# Assemble the complete traced agentic cycle and return a human-readable response plus the proposed action state.
@mlflow.trace(name="contract_intelligence_agent", span_type=SpanType.AGENT)
def run_agentic_cycle(question: str):
    plan = supervisor_agent(question)
    contract_context = contract_agent(question, plan)
    risk = risk_agent(question, contract_context)
    action = action_agent(question, contract_context, risk)

    facts = contract_context.get("structured_facts", [])
    first_fact = facts[0] if facts else {}

    final_response = (
        f"Contract: {action.get('contract_id') or 'portfolio-level'}\n"
        f"Supplier: {action.get('supplier_name') or first_fact.get('supplier_name', 'N/A')}\n"
        f"Risk: {risk.get('risk_type')} / {risk.get('risk_severity')}\n"
        f"Next milestone: {first_fact.get('next_milestone', 'N/A')}\n"
        f"Next milestone date: {first_fact.get('next_milestone_date', 'N/A')}\n"
        f"Next payment: {first_fact.get('next_payment_amount', 'N/A')} "
        f"{first_fact.get('next_payment_currency', first_fact.get('currency', ''))}\n"
        f"Payment due: {first_fact.get('next_payment_due_date', 'N/A')}\n"
        f"Delay penalty: {first_fact.get('delay_penalty_pct_per_week', 'N/A')}% per week\n"
        f"Recommended action: {action.get('proposed_action')}\n"
        f"Priority: {action.get('priority')}\n"
        f"Rationale: {action.get('rationale')}\n"
        f"Evidence: {action.get('evidence')}"
    )

    return {
        "final_response": final_response,
        "plan": plan,
        "contract_context": contract_context,
        "risk_assessment": risk,
        "proposed_action": action,
    }


In [0]:
# Provide the evaluation wrapper whose input signature matches the MLflow evaluation dataset.
def agent_for_evaluation(question: str):
    return run_agentic_cycle(question)["final_response"]


## Section 12 — Development Trace Test

Run one request before evaluation to inspect the full trace.

Example question:

> What is the main contract risk, next milestone, payment exposure, and delay penalty for this contract, and what follow-up should be taken?


In [0]:
# Run one development trace through the complete Supervisor-to-Action-Agent flow without creating an action-queue record.
sample_contract = (
    spark.table(GOLD_TABLE)
    .filter(F.col("risk_severity") == "HIGH")
    .select("contract_id")
    .limit(1)
    .collect()[0]["contract_id"]
)

development_question = (
    f"For contract {sample_contract}, identify the main risk, "
    "next milestone, payment exposure, delay penalty, "
    "and the recommended follow-up action."
)

development_result = run_agentic_cycle(development_question)

print(development_result["final_response"])


## Section 13 — Build the Pre-Deployment Evaluation Dataset

The evaluation set is generated from Lab A ground truth.

Each row contains:
- `inputs.question`
- `expectations.expected_facts`

Using `expected_facts` allows `Correctness` and `RetrievalSufficiency` to judge semantic factual coverage rather than exact wording.


In [0]:
# Build a compact ground-truth evaluation dataset from current contract data and clause evidence.
evaluation_source_rows = (
    spark.table(GOLD_TABLE)
    .filter(F.col("risk_severity").isin("HIGH", "MEDIUM"))
    .orderBy("contract_id")
    .limit(4)
    .collect()
)

eval_dataset = []

for row in evaluation_source_rows:
    r = row.asDict(recursive=True)

    question = (
        f"For contract {r['contract_id']}, identify the supplier, "
        "main risk and severity, next milestone, next payment exposure, "
        "delay penalty, and appropriate follow-up."
    )

    expected_facts = [
        f"Contract ID is {r['contract_id']}",
        f"Supplier is {r['supplier_name']}",
        f"Risk type is {r['risk_type']}",
        f"Risk severity is {r['risk_severity']}",
        f"Next milestone is {r['next_milestone']}",
        f"Next payment amount is {r['next_payment_amount']} {r['next_payment_currency']}",
        f"Delay penalty is {r['delay_penalty_pct_per_week']} percent per week",
        f"Recommended action is {r['recommended_action']}",
    ]

    eval_dataset.append(
        {
            "inputs": {
                "question": question
            },
            "expectations": {
                "expected_facts": expected_facts
            },
        }
    )

print("Evaluation examples:", len(eval_dataset))
print(json.dumps(eval_dataset[0], indent=2, default=str))


## Section 14 — Run the Required MLflow GenAI Evaluations

Required judges:

```python
RetrievalGroundedness()
RetrievalRelevance()
Correctness()
RetrievalSufficiency()
```

The retrieval judges inspect the `RETRIEVER` span created by `retrieve_contract_evidence(...)`.

The Databricks-backed judge model is requested explicitly.


In [0]:
# Run pre-deployment MLflow GenAI evaluation with groundedness, relevance, correctness, and retrieval sufficiency judges.
EVALUATION_COMPLETED = False
evaluation_results = None

try:
    evaluation_results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=agent_for_evaluation,
        scorers=[
            RetrievalGroundedness(model="databricks"),
            RetrievalRelevance(model="databricks"),
            Correctness(model="databricks"),
            RetrievalSufficiency(model="databricks"),
        ],
    )

    EVALUATION_COMPLETED = True

    print("Evaluation run ID:", evaluation_results.run_id)
    print("Aggregated metrics:")
    print(json.dumps(evaluation_results.metrics, indent=2, default=str))

    display(evaluation_results.result_df)

except Exception as e:
    print("MLflow GenAI evaluation could not complete.")
    print(type(e).__name__, str(e)[:1200])


## Section 15 — Apply the Deployment Quality Gate

The notebook searches the returned MLflow metrics for either:
- `/pass_rate`
- `/mean`

A candidate is deployment-ready only if:
- all four required evaluation families are present,
- every required metric meets the configured threshold.

For this small POC the threshold is **0.75**.

Increase this for a production implementation.


In [0]:
# Convert MLflow evaluation metrics into an explicit deployment-readiness gate.
MIN_EVAL_SCORE = 0.75

def _metric_for_family(metrics, family):
    preferred_keys = [
        f"{family}/pass_rate",
        f"{family}/mean",
    ]

    for key in preferred_keys:
        if key in metrics:
            return key, float(metrics[key])

    for key, value in metrics.items():
        if family in key:
            try:
                return key, float(value)
            except Exception:
                pass

    return None, None

required_metric_families = [
    "retrieval_groundedness",
    "retrieval_relevance",
    "correctness",
    "retrieval_sufficiency",
]

gate_rows = []

if EVALUATION_COMPLETED:
    for family in required_metric_families:
        metric_key, metric_value = _metric_for_family(
            evaluation_results.metrics,
            family
        )

        gate_rows.append(
            {
                "evaluation": family,
                "metric_key": metric_key,
                "score": metric_value,
                "threshold": MIN_EVAL_SCORE,
                "passed": (
                    metric_value is not None
                    and metric_value >= MIN_EVAL_SCORE
                ),
            }
        )
else:
    for family in required_metric_families:
        gate_rows.append(
            {
                "evaluation": family,
                "metric_key": None,
                "score": None,
                "threshold": MIN_EVAL_SCORE,
                "passed": False,
            }
        )

evaluation_gate_pdf = pd.DataFrame(gate_rows)

display(evaluation_gate_pdf)

DEPLOYMENT_READY = bool(
    EVALUATION_COMPLETED
    and evaluation_gate_pdf["passed"].all()
)

print("DEPLOYMENT_READY =", DEPLOYMENT_READY)


## Section 16 — Development Override

For an actual deployment, leave this `False`.

If Free Edition cannot access the judge model or evaluation service, you may set the development override to `True` only to demonstrate the downstream Human Approval → Action Queue mechanics.

The override does **not** change `DEPLOYMENT_READY`.


In [0]:
# Keep the deployment gate strict while allowing an explicit development-only demonstration override.
DEVELOPMENT_DEMO_OVERRIDE = False

CAN_RUN_OPERATIONAL_DEMO = (
    DEPLOYMENT_READY
    or DEVELOPMENT_DEMO_OVERRIDE
)

print("Deployment ready:", DEPLOYMENT_READY)
print("Development override:", DEVELOPMENT_DEMO_OVERRIDE)
print("Operational demo enabled:", CAN_RUN_OPERATIONAL_DEMO)


## Section 17 — Create the Human Approval Record

The Action Agent's proposal is persisted with:

`approval_status = PENDING`

No action is queued yet.


In [0]:
# Create a pending human-approval record from an agent proposal without writing anything to the action queue.
def create_approval_request(agent_result: dict):
    action = agent_result["proposed_action"]

    approval_id = str(uuid.uuid4())
    action_id = str(uuid.uuid4())

    approval_record = [
        (
            approval_id,
            action_id,
            action.get("contract_id"),
            action.get("supplier_name"),
            action.get("proposed_action"),
            action.get("priority"),
            action.get("rationale"),
            action.get("evidence"),
            datetime.utcnow(),
            "PENDING",
            None,
            None,
        )
    ]

    approval_df = spark.createDataFrame(
        approval_record,
        """
        approval_id string,
        action_id string,
        contract_id string,
        supplier_name string,
        proposed_action string,
        priority string,
        rationale string,
        evidence string,
        requested_at timestamp,
        approval_status string,
        approver string,
        reviewed_at timestamp
        """
    )

    approval_df.write.mode("append").format("delta").saveAsTable(
        APPROVAL_TABLE
    )

    return approval_id

print("Human approval function ready.")


## Section 18 — Human Decision and Action Queue

The human can choose:

- `APPROVE`
- `REJECT`

Only `APPROVE` writes the proposal into the action queue.


In [0]:
# Apply the human decision and enqueue the action only when the proposal is explicitly approved.
def review_and_enqueue(
    approval_id: str,
    decision: str,
    approver: str = "POC Human Reviewer"
):
    decision = decision.upper().strip()

    if decision not in {"APPROVE", "REJECT"}:
        raise ValueError(
            "decision must be APPROVE or REJECT"
        )

    status = (
        "APPROVED"
        if decision == "APPROVE"
        else "REJECTED"
    )

    spark.sql(
        f"""
        UPDATE {APPROVAL_TABLE}
        SET
            approval_status = '{status}',
            approver = '{approver.replace("'", "''")}',
            reviewed_at = current_timestamp()
        WHERE approval_id = '{approval_id}'
          AND approval_status = 'PENDING'
        """
    )

    reviewed = (
        spark.table(APPROVAL_TABLE)
        .filter(F.col("approval_id") == approval_id)
        .limit(1)
        .collect()
    )

    if not reviewed:
        raise RuntimeError(
            "Approval record not found."
        )

    record = reviewed[0].asDict(recursive=True)

    if record["approval_status"] != "APPROVED":
        return {
            "approval_status": record["approval_status"],
            "queued": False,
        }

    existing_queue_rows = (
        spark.table(ACTION_QUEUE_TABLE)
        .filter(F.col("action_id") == record["action_id"])
        .count()
    )

    if existing_queue_rows == 0:
        queue_df = spark.createDataFrame(
            [
                (
                    record["action_id"],
                    record["approval_id"],
                    record["contract_id"],
                    record["supplier_name"],
                    "CONTRACT_FOLLOW_UP",
                    record["proposed_action"],
                    record["priority"],
                    record["rationale"],
                    record["evidence"],
                    "QUEUED",
                    datetime.utcnow(),
                    None,
                )
            ],
            """
            action_id string,
            approval_id string,
            contract_id string,
            supplier_name string,
            action_type string,
            proposed_action string,
            priority string,
            rationale string,
            evidence string,
            queue_status string,
            queued_at timestamp,
            completed_at timestamp
            """
        )

        queue_df.write.mode("append").format("delta").saveAsTable(
            ACTION_QUEUE_TABLE
        )

    return {
        "approval_status": "APPROVED",
        "queued": True,
        "action_id": record["action_id"],
    }


## Section 19 — Execute the Full POC Cycle

The normal sequence is:

```text
Agent recommendation
      ↓
Evaluation gate
      ↓
Human approval request
      ↓
Human reviews evidence
      ↓
APPROVE / REJECT
      ↓
Approved action enters queue
```

For the POC, the human decision is represented by a variable.

Change:

```python
HUMAN_DECISION = "APPROVE"
```

to `"REJECT"` to test the alternative path.


In [0]:
# Run the complete operational POC after the quality gate and create a pending human-approval request.
operational_question = (
    f"For contract {sample_contract}, identify the current contract risk, "
    "milestone and payment exposure, supporting contract evidence, "
    "and propose the follow-up action."
)

operational_agent_result = None
approval_id = None

if CAN_RUN_OPERATIONAL_DEMO:
    operational_agent_result = run_agentic_cycle(
        operational_question
    )

    print(
        operational_agent_result[
            "final_response"
        ]
    )

    approval_id = create_approval_request(
        operational_agent_result
    )

    print("Approval request created:", approval_id)

else:
    print(
        "Operational cycle stopped because the pre-deployment "
        "evaluation gate has not passed."
    )


In [0]:
# Simulate the explicit human approval decision and move only approved proposals into the action queue.
HUMAN_DECISION = "APPROVE"

if approval_id is not None:
    human_result = review_and_enqueue(
        approval_id=approval_id,
        decision=HUMAN_DECISION,
        approver="POC Human Reviewer"
    )

    print(human_result)

else:
    print(
        "No approval record is available because the operational "
        "cycle was not enabled."
    )


## Section 20 — Inspect Human Approvals and the Action Queue

In [0]:
# Display the approval audit trail and the final action queue created by the agentic workflow.
print("HUMAN APPROVAL AUDIT")
display(
    spark.table(APPROVAL_TABLE)
    .orderBy(
        F.col("requested_at").desc()
    )
)

print("ACTION QUEUE")
display(
    spark.table(ACTION_QUEUE_TABLE)
    .orderBy(
        F.col("queued_at").desc()
    )
)


## Section 21 — What Each Agent Owns

| Component | Responsibility | Can execute enterprise action? |
|---|---|---:|
| Supervisor Agent | Understand request and route work | No |
| Contract Agent | Gather structured facts + contract evidence | No |
| Risk Agent | Interpret contract exposure | No |
| Action Agent | Propose follow-up | No |
| MLflow Evaluation Gate | Test retrieval and answer quality | No |
| Human Approver | Authorize or reject proposed action | Yes — authorization only |
| Action Queue | Persist approved follow-up for downstream execution | Downstream system executes |

### Separation of concerns

**Structured facts**
- contract value
- next milestone
- payment date
- renewal timing
- current risk metadata

**Retrieved evidence**
- SLA clauses
- penalties
- payment conditions
- renewal / termination clauses

**Reasoning**
- risk interpretation
- intervention rationale

**Action**
- supplier follow-up
- procurement escalation
- payment validation
- renewal initiation

**Control**
- evaluation gate
- human approval
- action audit trail


## Section 22 — MLflow Evaluation Logic

### RetrievalRelevance
Tests whether the retrieved clauses are relevant to the user's contract question.

### RetrievalGroundedness
Tests whether the agent's claims are supported by the retrieved contract evidence.

### RetrievalSufficiency
Uses `expected_facts` and asks whether the retrieved evidence contains enough information to support the expected answer.

### Correctness
Compares the final response against known expected contract facts.

### Why evaluate before deployment?

```text
Agent code
   ↓
Evaluation dataset
   ↓
Agent traces
   ↓
Retrieval + response judges
   ↓
Quality gate
   ↓
Deployment readiness
```

This prevents a model or prompt change from going directly into the action workflow without regression testing.


## Section 23 — Full Agentic Lifecycle

```text
                    ┌─────────────────────┐
                    │ Enterprise Contract │
                    │ Foundation — Lab A  │
                    └──────────┬──────────┘
                               │
                               ▼
                        SUPERVISOR AGENT
                               │
                               ▼
                         CONTRACT AGENT
                         /                                    /                           Structured Tool       Retriever
                   │                  │
                   └────────┬─────────┘
                            ▼
                         RISK AGENT
                            │
                            ▼
                        ACTION AGENT
                            │
                            ▼
                  MLflow EVALUATION GATE
                            │
                  ┌─────────┴─────────┐
                  │                   │
                FAIL                PASS
                  │                   │
                  ▼                   ▼
              Improve app       HUMAN APPROVAL
                                      │
                              ┌───────┴───────┐
                              ▼               ▼
                           REJECT          APPROVE
                                              │
                                              ▼
                                         ACTION QUEUE
                                              │
                                              ▼
                                  Enterprise follow-up
                                              │
                                              ▼
                                       Outcome / trace
                                              │
                                              └──► future eval cases
```


## Section 24 — Certification and Architecture Takeaways

1. Agents should call governed tools instead of depending on one giant prompt.
2. Structured enterprise questions should use SQL / Delta facts.
3. Contractual interpretation should retrieve the underlying clause evidence.
4. Retrieval functions should be traced as `RETRIEVER` spans for RAG evaluation.
5. Tool calls and agent steps should be visible in the trace.
6. `Correctness` needs ground-truth expectations such as `expected_facts`.
7. `RetrievalSufficiency` also needs expected facts or an expected response.
8. Groundedness and correctness are different:
   - groundedness asks whether the answer is supported by context,
   - correctness asks whether it matches known truth.
9. Retrieval relevance and retrieval sufficiency are different:
   - relevance asks whether retrieved chunks relate to the request,
   - sufficiency asks whether enough information was retrieved.
10. A quality gate should run before deployment.
11. Human approval should remain outside autonomous model reasoning for consequential actions.
12. The action queue creates an auditable boundary between AI recommendation and enterprise execution.
13. Traces, human decisions, and production failures can become future evaluation examples.


In [0]:
# Print the final Lab B components and operational status for easy verification.
lab_b_summary = {
    "flow": [
        "Supervisor Agent",
        "Contract Agent",
        "Risk Agent",
        "Action Agent",
        "MLflow Evaluation Gate",
        "Human Approval",
        "Action Queue",
    ],
    "required_evaluations": [
        "RetrievalGroundedness",
        "RetrievalRelevance",
        "Correctness",
        "RetrievalSufficiency",
    ],
    "approval_table": APPROVAL_TABLE,
    "action_queue_table": ACTION_QUEUE_TABLE,
    "evaluation_completed": EVALUATION_COMPLETED,
    "deployment_ready": DEPLOYMENT_READY,
    "development_demo_override": DEVELOPMENT_DEMO_OVERRIDE,
}

print(json.dumps(lab_b_summary, indent=2))


## Lab B Completion Checklist

### Agentic cycle
- Supervisor Agent implemented
- Contract Agent implemented
- Risk Agent implemented
- Action Agent implemented
- Human approval boundary implemented
- Action Queue implemented

### Tools
- Structured contract tool traced as `TOOL`
- Contract evidence retrieval traced as `RETRIEVER`
- LLM calls traced as `LLM`
- Agent functions traced as `AGENT`

### Evaluation before deployment
- Evaluation dataset generated from contract facts
- `RetrievalGroundedness()` configured
- `RetrievalRelevance()` configured
- `Correctness()` configured
- `RetrievalSufficiency()` configured
- Deployment readiness gate implemented

### Governance
- Agent cannot directly write to the action queue
- Approval begins as `PENDING`
- Only explicit `APPROVE` moves an action to the queue
- Approval and queue records are retained in Delta tables

### Overall POC

**Lab A**

`Enterprise sources → AI extraction → Delta contract intelligence → Enterprise view → Retrieval`

**Lab B**

`Supervisor → Contract Agent → Risk Agent → Action Agent → Evals → Human Approval → Action Queue`

Together, the two notebooks demonstrate an end-to-end **enterprise contract intelligence + agentic action lifecycle**.
